## Acemi — mevcut oyun kayıtlarının Stockfish analizi (Kaggle)

1. **Internet: On** (Settings → Internet → On) ve Kaggle Secret olarak `GITHUB_TOKEN` ekli olsun.
2. **Run All**. Tahmini süre: ~30–60 dk tam analiz + ~30–40 dk sentetik; oturum sınırına dikkat.
3. Hücre sırası: repo/Stockfish kurulumu → süre tahmini → 1 oyunluk smoke → tam analiz (parça parça GitHub'a push) → depth-4 recheck → sentetik oyun sonları → `REPORT.md`.
4. Oturum kesilirse aynı notebook'u baştan çalıştır: `data/analysis/cache` ve `processed_games.txt` sayesinde kaldığı yerden devam eder.
5. Sonuçlar `data/analysis/` altında GitHub'a push edilir; ara cache dosyaları repoya girmez.


In [ ]:
# Repository and output configuration.
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "main"
WORKSPACE = "/kaggle/working/acemi"
ANALYSIS_DIR = "data/analysis"


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

def run_quietly(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        details = "\n".join(part for part in (result.stdout, result.stderr) if part)
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{details[-4000:]}")
    return result.stdout

repo_url = GITHUB_REPO_URL.strip()
workspace = Path(WORKSPACE)
if not workspace.exists():
    run_quietly(["git", "clone", "--branch", GITHUB_BRANCH, repo_url, str(workspace)])
else:
    run_quietly(["git", "-C", str(workspace), "pull", "--ff-only", "origin", GITHUB_BRANCH])
os.chdir(workspace)
print(run_quietly(["git", "-C", str(workspace), "log", "-1", "--format=%h %s"]).strip())
print("Engine version:", subprocess.run(["python", "-c", "import engine; print(engine.__version__)"], text=True, capture_output=True).stdout.strip())


In [ ]:
run_quietly(["apt-get", "update", "-qq"])
run_quietly(["apt-get", "install", "-y", "-qq", "stockfish"])
run_quietly(["python", "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"])
stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    raise RuntimeError("Stockfish package installed but executable was not found.")
version = subprocess.run([stockfish_bin, "--version"], check=True, text=True, capture_output=True).stdout.splitlines()[0]
print("Stockfish:", version)
print("Analysis tools ready.")


In [ ]:
# Publish helper: commits data/analysis only (cache is gitignored) and pushes.
import base64
from kaggle_secrets import UserSecretsClient

def publish_analysis(message):
    try:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is unavailable; add it under Add-ons → Secrets.") from None
    if not token:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty.")
    auth_value = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": auth_value,
        "GIT_TERMINAL_PROMPT": "0",
    })
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=git_env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stderr + result.stdout).replace(token, "[REDACTED]")
            raise RuntimeError(f"Git command failed ({args[0]}): {details[-1500:]}")
        return result
    git("config", "user.name", "Acemi Analysis")
    git("config", "user.email", "acemi-analysis@users.noreply.github.com")
    Path(workspace / ANALYSIS_DIR).mkdir(parents=True, exist_ok=True)
    git("add", "--", ANALYSIS_DIR)
    if git("diff", "--cached", "--quiet", check=False).returncode == 0:
        print("Nothing new to publish.")
        return
    git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()) > 0:
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Estimate wall time from the manifest before spending the session.
workers = max(1, (os.cpu_count() or 2) - 1)
subprocess.run([
    "python", "-m", "tools.analyze_games", "--estimate",
    "--depth", "10", "--workers", str(workers), "--out", ANALYSIS_DIR,
], check=True)
print("workers =", workers)


In [ ]:
# Smoke test: one game only, separate output directory.
subprocess.run([
    "python", "-m", "tools.analyze_games", "--smoke", "--max-games", "1",
    "--workers", "1", "--depth", "10", "--stockfish", str(stockfish_bin),
    "--out", "data/analysis_smoke",
], check=True)
print("Smoke test finished.")


In [ ]:
# Full analysis in chunks; each chunk is published so a dead session can resume.
for chunk in range(1, 200):
    result = subprocess.run([
        "python", "-m", "tools.analyze_games",
        "--depth", "10", "--workers", str(workers),
        "--max-new-games", "250", "--stockfish", str(stockfish_bin),
        "--out", ANALYSIS_DIR,
    ], check=False)
    if result.returncode not in (0, 2):
        raise RuntimeError(f"Analysis failed with exit code {result.returncode}")
    publish_analysis(f"data: game analysis chunk {chunk}")
    if result.returncode == 2:
        print("All unique games processed.")
        break


In [ ]:
# Depth-4 recheck: can Acemi see its own blunders at depth 4?
subprocess.run([
    "python", "-m", "tools.analyze_games", "--recheck", "--out", ANALYSIS_DIR,
], check=True)
publish_analysis("data: depth-4 blunder recheck")


In [ ]:
# Synthetic endgames: KQK, KRK, KBBK, KBNK, KPK, KRPKR (SF-labelled expected results).
subprocess.run([
    "python", "-m", "tools.synthetic_endgames",
    "--stockfish", str(stockfish_bin),
    "--out", f"{ANALYSIS_DIR}/synthetic_endgames.csv",
    "--count", "10", "--movetime-ms", "500", "--max-plies", "80",
], check=True)
publish_analysis("data: synthetic endgame diagnostic")


In [ ]:
# Rebuild REPORT.md from the CSVs and publish.
subprocess.run(["python", "-m", "tools.analyze_report", "--dir", ANALYSIS_DIR], check=True)
publish_analysis("data: analysis report")


In [ ]:
# Final safety push.
publish_analysis("data: analysis final")
